Below we show how a KNN model handles and predicts our airbnb data, we have:
1. a model with all 69 features and no scaling
2. A model with all 69 features, but scaled for all features
3. A model where we use the optimal lasso alpha value found in the regression model to determine noisy features we can remove to improve the KNN's performance 
4. a model where  we set the alpha of the lasso to 1 to reduce even more features and see how that impacts the KNN's performance

 We used Lasso as a practical dimensionality reduction tool to address KNN's sensitivity to high-dimensional feature spaces. While Lasso's linear assumptions don't perfectly align with KNN's distance-based logic, it provides a principled, data-driven method for removing weak features, which empirically improved KNN performance significantly.

below is knn method with all features and scaling due to large range of values 

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('Use_this_data_for_Modeling_MSE446_cleaned_dataset_postEDA2.csv')
df.head()

X = df.drop('rate_avg', axis=1)
y = df['rate_avg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Scaling features (required for KNN — distance-based algorithm)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

neighbors_settings = [3, 4, 5, 7, 9, 11, ]

for n_neighbors in neighbors_settings:
    # Build the model
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    
    # Train the model
    knn.fit(X_train_scaled, y_train)
    
    # Make predictions on the test set
    y_pred = knn.predict(X_test_scaled)
    
    # Testing  the model
    # R-squared
    r2 = r2_score(y_test, y_pred)
    # Mean Squared Error (MSE)
    mse = mean_squared_error(y_test, y_pred)
    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse)
    
    print(f"R^2 for {n_neighbors} neighbors: {r2}")
    print(f"MSE for {n_neighbors} neighbors: {mse}")
    print(f"RMSE for {n_neighbors} neighbors: {rmse}")



(2580, 70) (645, 70) (2580,) (645,)
R^2 for 3 neighbors: 0.621595536509209
MSE for 3 neighbors: 4026.625333333333
RMSE for 3 neighbors: 63.45569583050313
R^2 for 4 neighbors: 0.6653624838521961
MSE for 4 neighbors: 3560.8985358527134
RMSE for 4 neighbors: 59.673264833195724
R^2 for 5 neighbors: 0.6785998473626038
MSE for 5 neighbors: 3420.0389308527133
RMSE for 5 neighbors: 58.48109891967415
R^2 for 7 neighbors: 0.7022024782744698
MSE for 7 neighbors: 3168.881873437747
RMSE for 7 neighbors: 56.29282257479853
R^2 for 9 neighbors: 0.715730535603425
MSE for 9 neighbors: 3024.9289775098096
RMSE for 9 neighbors: 54.999354337208445
R^2 for 11 neighbors: 0.7200511091343679
MSE for 11 neighbors: 2978.95348696265
RMSE for 11 neighbors: 54.57979009635938


Below is a KNN model without scaling to show why it was necessary....

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv('Use_this_data_for_Modeling_MSE446_cleaned_dataset_postEDA2.csv')
df.head()

X = df.drop('rate_avg', axis=1)
y = df['rate_avg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

neighbors_settings = [3, 4, 5, 7, 9, 11]

for n_neighbors in neighbors_settings:
    # Building the model
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    
    # Training the model
    knn.fit(X_train, y_train)
    
    # Making predictions on the test set
    y_pred = knn.predict(X_test)
    
    # Test the model
    # R-squared
    r2 = r2_score(y_test, y_pred)
    # Mean Squared Error (MSE)
    mse = mean_squared_error(y_test, y_pred)
    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse) 
    
    print(f"R^2 for {n_neighbors} neighbors: {r2}")
    print(f"MSE for {n_neighbors} neighbors: {mse}")
    print(f"RMSE for {n_neighbors} neighbors: {rmse}")

(2580, 70) (645, 70) (2580,) (645,)
R^2 for 3 neighbors: 0.19282532988027545
MSE for 3 neighbors: 8589.195658914727
RMSE for 3 neighbors: 92.67791354424595
R^2 for 4 neighbors: 0.21651745730475336
MSE for 4 neighbors: 8337.08626356589
RMSE for 4 neighbors: 91.30764624918271
R^2 for 5 neighbors: 0.22137822099045723
MSE for 5 neighbors: 8285.362576124031
RMSE for 5 neighbors: 91.02396704233469


R^2 for 7 neighbors: 0.2234605191992306
MSE for 7 neighbors: 8263.204711912673
RMSE for 7 neighbors: 90.902171106705
R^2 for 9 neighbors: 0.24840176290380078
MSE for 9 neighbors: 7997.803392860561
RMSE for 9 neighbors: 89.43043884975943
R^2 for 11 neighbors: 0.24804623361073852
MSE for 11 neighbors: 8001.586602088539
RMSE for 11 neighbors: 89.45158803558793


Knn without scaling and with all 69 features is much higher (essentially double) compared to when scaling the range of our features, eliminating higher value bias

Below is a KNN model with scalar, with feature selection only considering features deemed necessary by Lasso. 
1. determine the optimal alpha value via CV
2. use that alpha value in lasso 
3. run KNN model with only non-zeroed features

Below an arbitrary value of alpha of 1 is used to see how the lasso feature selection impacts the KNN models.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
 
df = pd.read_csv('Use_this_data_for_Modeling_MSE446_cleaned_dataset_postEDA2.csv')
df.head()
 
X = df.drop('rate_avg', axis=1)
y = df['rate_avg']
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
 
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
 
# Scaling features (required for Lasso — penalizes coefficients, so features must be on same scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 
# Building and training the model
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
 
# Making predictions on the test set
y_pred = lasso.predict(X_test_scaled)
 

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
 
print(f"R^2: {r2}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
 
# Identify which features were zeroed out by Lasso
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': lasso.coef_})
 
zeroed_features = coef_df[coef_df['coefficient'] == 0]
kept_features = coef_df[coef_df['coefficient'] != 0]
 
print(f"\nTotal features: {len(X.columns)}")
print(f"Features zeroed out by Lasso: {len(zeroed_features)}")
print(f"Features kept by Lasso: {len(kept_features)}")
 
print(f"\nZeroed out features:")
print(zeroed_features['feature'].tolist())
 
print(f"\nKept features and their coefficients:")
print(kept_features.sort_values('coefficient', key=abs, ascending=False).to_string(index=False))

(2580, 70) (645, 70) (2580,) (645,)
R^2: 0.7184068469284223
MSE: 2996.450182937179
RMSE: 54.73984091077703

Total features: 70
Features zeroed out by Lasso: 31
Features kept by Lasso: 39

Zeroed out features:
['rating_overall', 'rating_accuracy', 'rating_checkin', 'rating_cleanliness', 'rating_communication', 'rating_value', 'Hotel_Occupancy_Percentage', 'Average_Daily_Rate', 'Revenue_Per_Available_Room', 'is_winter', 'is_spring', 'is_fall', 'num_public_holidays', 'num_major_events', 'month', 'is_april', 'is_august', 'is_february', 'is_july', 'is_june', 'is_may', 'is_november', 'is_october', 'has_gym', 'listing_type_Entire house', 'listing_type_Private room in bungalow', 'listing_type_Private room in cottage', 'listing_type_Private room in home', 'listing_type_Private room in loft', 'room_type_private_room', 'cancellation_policy_Super Strict 30 Days']

Kept features and their coefficients:
                                 feature  coefficient
                                bedrooms   

KNN model without 31 zerod out features by Lasso:

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv('Use_this_data_for_Modeling_MSE446_cleaned_dataset_postEDA2.csv')
df.head()

X = df.drop(['rate_avg', 'rating_overall', 'rating_accuracy', 'rating_checkin', 
             'rating_cleanliness', 'rating_communication', 'rating_value', 
             'Hotel_Occupancy_Percentage', 'Average_Daily_Rate', 'Revenue_Per_Available_Room', 
             'is_winter', 'is_spring', 'is_fall', 'num_public_holidays', 'num_major_events', 
             'month', 'is_april', 'is_august', 'is_february', 'is_july', 'is_june', 'is_may', 
             'is_november', 'is_october', 'has_gym', 'listing_type_Entire house', 
             'listing_type_Private room in bungalow', 'listing_type_Private room in cottage', 
             'listing_type_Private room in home', 'listing_type_Private room in loft', 
             'room_type_private_room', 'cancellation_policy_Super Strict 30 Days'], axis=1)
y = df['rate_avg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

neighbors_settings = [3, 4, 5, 7, 9, 11]

for n_neighbors in neighbors_settings:
    # Builing the model
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    
    # Training the model
    knn.fit(X_train_scaled, y_train)
    
    # Making predictions on the test set
    y_pred = knn.predict(X_test_scaled)
    
    # Testing the model
    # R-squared
    r2 = r2_score(y_test, y_pred)
    # Mean Squared Error (MSE)
    mse = mean_squared_error(y_test, y_pred)
    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse) 
    
    print(f"R^2 for {n_neighbors} neighbors: {r2}")
    print(f"MSE for {n_neighbors} neighbors: {mse}")
    print(f"RMSE for {n_neighbors} neighbors: {rmse}")

(2580, 39) (645, 39) (2580,) (645,)
R^2 for 3 neighbors: 0.8887850554941146
MSE for 3 neighbors: 1183.4451128337641
RMSE for 3 neighbors: 34.40123708289811
R^2 for 4 neighbors: 0.8995109975912022
MSE for 4 neighbors: 1069.309698643411
RMSE for 4 neighbors: 32.70030120111145
R^2 for 5 neighbors: 0.8924655336132582
MSE for 5 neighbors: 1144.2809172093023
RMSE for 5 neighbors: 33.82722154137555


R^2 for 7 neighbors: 0.8660414129730479
MSE for 7 neighbors: 1425.461621578864
RMSE for 7 neighbors: 37.755286008436805
R^2 for 9 neighbors: 0.8484963223484827
MSE for 9 neighbors: 1612.1600176093407
RMSE for 9 neighbors: 40.151712511539785
R^2 for 11 neighbors: 0.8255598185943045
MSE for 11 neighbors: 1856.2287746812738
RMSE for 11 neighbors: 43.08397352474901


Below is the KNN model with reduced features, via Lasso to counteract uncorrelated noise: The Lasso alpha value of 0.035 resulted in the removal of 17 features which we will remove from the original KNN model to see if removing Lasso's deemed noisy features will improve the model's performance. - refactor below to only include KNN model part with the 17 reduced found in Regression model findings

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv('Use_this_data_for_Modeling_MSE446_cleaned_dataset_postEDA2.csv')
df.head()

X = df.drop(['rate_avg', 'Hotel_Occupancy_Percentage', 'Revenue_Per_Available_Room', 'is_winter', 'is_spring', 'is_summer', 'is_fall', 'num_public_holidays', 'non_resident_visitors', 'is_august', 'is_february', 'is_july', 'is_june', 'is_november', 'is_september', 'listing_type_Private room in bungalow', 'listing_type_Private room in home', 'listing_type_Private room in loft'], axis=1)
y = df['rate_avg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

neighbors_settings = [3, 4, 5, 7, 9, 11]

for n_neighbors in neighbors_settings:
    # Build the model
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    
    # Train the model
    knn.fit(X_train_scaled, y_train)
    
    # Make predictions on the test set
    y_pred = knn.predict(X_test_scaled)
    
    # Test the model
    # R-squared
    r2 = r2_score(y_test, y_pred)
    # Mean Squared Error (MSE)
    mse = mean_squared_error(y_test, y_pred)
    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse) 
    
    print(f"R^2 for {n_neighbors} neighbors: {r2}")
    print(f"MSE for {n_neighbors} neighbors: {mse}")
    print(f"RMSE for {n_neighbors} neighbors: {rmse}")

(2580, 53) (645, 53) (2580,) (645,)
R^2 for 3 neighbors: 0.8591049429778308
MSE for 3 neighbors: 1499.2730284237725
RMSE for 3 neighbors: 38.72044716198113
R^2 for 4 neighbors: 0.8531074065448924
MSE for 4 neighbors: 1563.0931850775196
RMSE for 4 neighbors: 39.53597330378398
R^2 for 5 neighbors: 0.8441857834808995
MSE for 5 neighbors: 1658.0287286821706
RMSE for 5 neighbors: 40.718898912939316
R^2 for 7 neighbors: 0.8440500047456103
MSE for 7 neighbors: 1659.4735586141435
RMSE for 7 neighbors: 40.73663656481894
R^2 for 9 neighbors: 0.8255308126932774
MSE for 9 neighbors: 1856.5374282706478
RMSE for 9 neighbors: 43.087555375893025
R^2 for 11 neighbors: 0.8087321664435098
MSE for 11 neighbors: 2035.2928634761995
RMSE for 11 neighbors: 45.11422019137868


Here the trend seems to be that even removing more than the features removed at the optimal Lasso, still improves the MSE and error metrics, so what's to say removing even more features will not further imrpove error, but then not provide any informative conclusion....